In [0]:
# %sql
# -- 1. Aseguramos que la tabla no tenga basura previa
# DROP TABLE IF EXISTS products.bronze_scraped_products;

In [0]:
%sql
-- 1. Crear la tabla SOLO si no existe
-- Usamos TBLPROPERTIES para soportar nombres de columnas con espacios (como tus 'Opcion 1')
CREATE TABLE IF NOT EXISTS workspace.products.bronze_scraped_products
USING DELTA
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
);

In [0]:
# %sql
# -- 2. Cargar datos de forma incremental
# -- COPY INTO es inteligente: mantiene un registro de qué archivos ya procesó
# COPY INTO workspace.products.bronze_scraped_products
# FROM '/Volumes/workspace/products/products_tracker'
# FILEFORMAT = JSON
# FORMAT_OPTIONS (
#   'multiLine' = 'true',      -- Necesario por el indent=4 de tu Python
#   'inferSchema' = 'true',
#   'mergeSchema' = 'true'     -- Permite que la tabla evolucione si agregas campos
# )
# COPY_OPTIONS (
#   'mergeSchema' = 'true',
#   'force' = 'false'          -- 'false' es el valor por defecto: SOLO carga archivos nuevos
# );

In [0]:
%sql
COPY INTO workspace.products.bronze_scraped_products
FROM (
  SELECT 
    to_timestamp(scraped_at) AS scraped_at,
    product_id,
    retailer,
    raw_data.sku AS sku,
    raw_data.name AS name,
    raw_data.brand AS brand,
    raw_data.main_category AS main_category,
    raw_data.sub_category AS sub_category,
    raw_data.list_price AS list_price,
    raw_data.cash_price AS cash_price,
    raw_data.stock AS stock,
    to_json(raw_data.installments) AS installments_json,
    _metadata.file_path AS file_metadata_path
  FROM '/Volumes/workspace/products/products_tracker/scraped'
)
FILEFORMAT = JSON
FORMAT_OPTIONS (
    'multiLine' = 'true',
    'inferSchema' = 'true'
)
COPY_OPTIONS (
    'mergeSchema' = 'true',
    'force' = 'false'
);